In [1]:
!pip install -q groq python-dotenv pydantic ollama chromadb langchain-text-splitters tiktoken


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import time
import requests
import ollama
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [2]:
load_dotenv()
cliente_groq = Groq()
MODELO_GROQ = "qwen/qwen3.8-27b"
MODELO_OLLAMA = "qwen3:8b"
BACKEND_ACTUAL = "ollama"
MODELO_EMBEDDINGS = "qwen3-embedding:8b"
RAG_CHUNK_SIZE = 1000
RAG_OVERLAP_PCT = 80
RAG_ESTRUCTURADO = True

In [3]:
funcion_embeddings = OllamaEmbeddingFunction(
    model_name=MODELO_EMBEDDINGS,
    url="http://localhost:11434/api/embeddings"
)

cliente_chroma = chromadb.PersistentClient(path="./chroma_db")

coleccion_politicas = cliente_chroma.get_or_create_collection(
    name="politicas_ecommerce",
    embedding_function=funcion_embeddings
)

In [4]:
try:
    with open("prompt_urgencia.md", "r", encoding="utf-8") as f:
        prompt_urgencia = f.read()
    with open("prompt_categoria.md", "r", encoding="utf-8") as f:
        prompt_categoria = f.read()
    with open("prompt_accion.md", "r", encoding="utf-8") as f:
        prompt_accion = f.read()
    with open("prompt_respuesta.md", "r", encoding="utf-8") as f:
        prompt_respuesta = f.read()
except FileNotFoundError as e:
    raise RuntimeError(f"Error al leer los archivos de prompt: {e}")

try:
    with open("politicas_ecommerce_integral.md", "r", encoding="utf-8") as f:
        politicas_ecommerce = f.read()
except FileNotFoundError as e:
    raise RuntimeError(f"Error al leer las políticas: {e}")

In [5]:
class Metadatos(BaseModel):
    categoria: str
    nivel_urgencia: str

class Analisis(BaseModel):
    cadena_de_pensamiento: str
    politica_aplicada: str

class AccionRespuesta(BaseModel):
    codigo_accion_sistema: str
    mensaje_al_cliente: str

class SalidaMaestra(BaseModel):
    metadatos_clasificacion: Metadatos
    analisis_interno: Analisis
    accion_y_respuesta: AccionRespuesta

In [6]:
def llamar_llm(system_prompt: str, input_data: dict) -> dict:
    mensajes = [
        {
            "role": "system",
            "content": f"{system_prompt}\n\nDebes responder ÚNICAMENTE con un objeto JSON válido."
        },
        {
            "role": "user",
            "content": json.dumps(input_data)
        }
    ]

    if BACKEND_ACTUAL == "groq":
        chat_completion = cliente_groq.chat.completions.create(
            messages=mensajes,
            model=MODELO_GROQ,
            response_format={"type": "json_object"},
            temperature=0.1,
        )
        return json.loads(chat_completion.choices[0].message.content)

    elif BACKEND_ACTUAL == "ollama":
        respuesta = ollama.chat(
            model=MODELO_OLLAMA,
            messages=mensajes,
            format="json",
            options={"temperature": 0.1}
        )
        return json.loads(respuesta['message']['content'])

In [ ]:
def indexar_politicas():
    try:
        ids_existentes = set(coleccion_politicas.get()['ids'])
    except Exception:
        ids_existentes = set()

    print(f"Documentos ya existentes en la base de datos: {len(ids_existentes)}")

    headers_to_split_on = [
        ("#", "Titulo_Principal"),
        ("##", "Seccion"),
        ("###", "Subseccion"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    splits_estructurales = markdown_splitter.split_text(politicas_ecommerce)

    tamanos = [10, 100, 1000]
    solapamientos_pct = [0.0, 0.1, 0.8]

    textos_batch = []
    metadatos_batch = []
    ids_batch = []

    for size in tamanos:
        for pct in solapamientos_pct:
            overlap_abs = int(size * pct)
            overlap_label = int(pct * 100)

            text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="cl100k_base",
                chunk_size=size,
                chunk_overlap=overlap_abs
            )

            # --- Fragmentos Estructurales ---
            fragmentos_estructurales = text_splitter.split_documents(splits_estructurales)
            for i, doc in enumerate(fragmentos_estructurales):
                doc_id = f"chunk_s{size}_o{overlap_label}_est_{i}"

                if doc_id not in ids_existentes:
                    meta = doc.metadata.copy()
                    meta.update({"chunk_size": size, "overlap_pct": overlap_label, "estructurado": True})
                    textos_batch.append(doc.page_content)
                    metadatos_batch.append(meta)
                    ids_batch.append(doc_id)

            # --- Fragmentos Planos ---
            fragmentos_planos = text_splitter.create_documents([politicas_ecommerce])
            for i, doc in enumerate(fragmentos_planos):
                doc_id = f"chunk_s{size}_o{overlap_label}_plano_{i}"

                if doc_id not in ids_existentes:
                    meta = doc.metadata.copy()
                    meta.update({"chunk_size": size, "overlap_pct": overlap_label, "estructurado": False})
                    textos_batch.append(doc.page_content)
                    metadatos_batch.append(meta)
                    ids_batch.append(doc_id)

    total_faltantes = len(textos_batch)
    if total_faltantes == 0:
        print("Todos los documentos ya estaban indexados. ¡Nada que hacer!")
        return

    print(f"Iniciando indexación de {total_faltantes} fragmentos FALTANTES con timeout ampliado...")

    for idx in range(total_faltantes):
        texto = textos_batch[idx]
        meta = metadatos_batch[idx]
        doc_id = ids_batch[idx]

        print(f"\nProcesando {idx+1}/{total_faltantes}: {doc_id}...")

        exito = False
        reintentos = 3

        while not exito and reintentos > 0:
            try:
                respuesta = requests.post(
                    "http://localhost:11434/api/embed",
                    json={"model": MODELO_EMBEDDINGS, "input": texto},
                    timeout=300
                )

                if respuesta.status_code == 200:
                    vector = respuesta.json()["embeddings"][0]

                    coleccion_politicas.upsert(
                        documents=[texto],
                        metadatas=[meta],
                        ids=[doc_id],
                        embeddings=[vector]
                    )
                    exito = True
                    print(f"✅ ¡Éxito! Insertado correctamente.")

                    time.sleep(2)

                else:
                    reintentos -= 1
                    print(f"❌ Error de Ollama: {respuesta.status_code} | {respuesta.text}")
                    print(f"Reintentos restantes: {reintentos}. Esperando 10 segundos...")
                    time.sleep(10)

            except requests.exceptions.Timeout:
                reintentos -= 1
                print(f"❌ Timeout incluso con 5 minutos de espera. Reintentos restantes: {reintentos}.")
                time.sleep(15)

            except Exception as e:
                reintentos -= 1
                print(f"❌ Error inesperado: {e}. Reintentos restantes: {reintentos}.")
                time.sleep(15)

        if not exito:
            print(f"\n⚠️ Imposible procesar {doc_id}. SALTANDO ESTE FRAGMENTO para continuar con el resto.")
            continue

    print(f"\nIndexación finalizada. Documentos totales en índice: {coleccion_politicas.count()}")

indexar_politicas()

In [7]:
def recuperar_contexto(mensaje_usuario: str, k: int) -> str:
    resultados = coleccion_politicas.query(
        query_texts=[mensaje_usuario],
        n_results=k,
        where={
            "$and": [
                {"chunk_size": {"$eq": RAG_CHUNK_SIZE}},
                {"overlap_pct": {"$eq": RAG_OVERLAP_PCT}},
                {"estructurado": {"$eq": RAG_ESTRUCTURADO}}
            ]
        }
    )

    documentos = resultados["documents"][0]
    metadatos = resultados["metadatas"][0]

    if not documentos:
        return "No se encontraron políticas relevantes."

    fragmentos_formateados = []

    for doc, meta in zip(documentos, metadatos):
        if meta.get("estructurado"):
            seccion = meta.get('Seccion', '')
            subseccion = meta.get('Subseccion', '')
            partes = [p for p in [seccion, subseccion] if p]

            if partes:
                encabezado = f"[{' -> '.join(partes)}]"
                fragmentos_formateados.append(f"{encabezado}\n{doc}")
            else:
                fragmentos_formateados.append(doc)
        else:
            fragmentos_formateados.append(doc)

    return "\n\n--- FRAGMENTO DE POLÍTICA ---\n\n".join(fragmentos_formateados)

In [8]:
def procesar_mensaje_con_rag(mensaje_usuario: str) -> dict:
    contexto_politicas = recuperar_contexto(mensaje_usuario, k=3)

    print(f"Contexto RAG recuperado. Longitud: {len(contexto_politicas)} caracteres.")

    def manejar_pausa():
        if BACKEND_ACTUAL == "groq":
            time.sleep(15)

    print(f"[{BACKEND_ACTUAL.upper()}] Procesando inferencias con LLM...")

    print("Ejecutando urgencia...")
    out_urgencia = llamar_llm(prompt_urgencia, {
        "mensaje_usuario": mensaje_usuario
    })
    nivel_urgencia = out_urgencia.get("nivel_urgencia", "")
    manejar_pausa()

    print("Ejecutando categoria...")
    out_categoria = llamar_llm(prompt_categoria, {
        "mensaje_usuario": mensaje_usuario
    })
    categoria = out_categoria.get("categoria", "")
    manejar_pausa()

    print("Ejecutando accion...")
    out_accion = llamar_llm(prompt_accion, {
        "mensaje_usuario": mensaje_usuario,
        "contexto_politicas": contexto_politicas
    })
    cadena_de_pensamiento = out_accion.get("cadena_de_pensamiento", "")
    politica_aplicada = out_accion.get("politica_aplicada", "")
    codigo_accion_sistema = out_accion.get("codigo_accion_sistema", "")
    manejar_pausa()

    print("Ejecutando respuesta...")
    out_texto = llamar_llm(prompt_respuesta, {
        "mensaje_usuario": mensaje_usuario,
        "politica_aplicada": politica_aplicada
    })
    mensaje_al_cliente = out_texto.get("mensaje_al_cliente", "")
    manejar_pausa()

    salida_estructurada = {
        "metadatos_clasificacion": {
            "categoria": categoria,
            "nivel_urgencia": nivel_urgencia
        },
        "analisis_interno": {
            "cadena_de_pensamiento": cadena_de_pensamiento,
            "politica_aplicada": politica_aplicada
        },
        "accion_y_respuesta": {
            "codigo_accion_sistema": codigo_accion_sistema,
            "mensaje_al_cliente": mensaje_al_cliente
        }
    }

    salida_validada = SalidaMaestra(**salida_estructurada)
    return salida_validada.model_dump()

In [9]:
# Caso borde 1
if __name__ == "__main__":
    mensaje = "¡Hola! Compré hace 11 días hábiles en pleno CyberMonday para mi domicilio en una provincia del interior. Como se demoraron más de los 8 días estándar que prometen, ayer salí a trabajar y el correo pasó justo y no encontró a nadie. Ahora me llega un aviso de que debo volver a pagar el envío. La demora fue de ustedes, exijo que me lo manden gratis o cancelen todo devolviéndome hasta el último centavo."
    try:
        resultado = procesar_mensaje_con_rag(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")

Contexto RAG recuperado. Longitud: 4952 caracteres.
[OLLAMA] Procesando inferencias con LLM...
Error procesando el mensaje: prediction aborted, token repeat limit reached (status code: 500)


In [ ]:
# Caso borde 2
if __name__ == "__main__":
    mensaje = "Buenas tardes, hace 70 días compré una licuadora. Ayer dejó de prender de la nada tras un corte de luz en mi barrio. Mi hermano, que es técnico de PC, le sacó los tornillos para ver si era un fusible quemado por la sobrecarga, pero me dijo que no. Como sigo dentro de los 90 días de la garantía, exijo el reemplazo directo del equipo hoy mismo."
    try:
        resultado = procesar_mensaje_con_rag(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")

In [ ]:
# Caso borde 3
if __name__ == "__main__":
    mensaje = "Estimados, hace exactamente 17 días hábiles hice una compra grande de electrodomésticos para mi oficina. Recién hoy la administración revisó la factura y me avisa que salió a mi nombre personal y no a nombre de la empresa. Les adjunto mi constancia de CUIT y la factura original en PDF para que me hagan una nota de crédito urgente y la refacturen correctamente, es solo un error de tipeo."
    try:
        resultado = procesar_mensaje_con_rag(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")

In [ ]:
# Grounding
if __name__ == "__main__":
    mensaje = "Hola. Ayer pasó el correo y no había nadie en mi domicilio para recibir el paquete. Exijo que lo traigan de nuevo mañana sin cargo, ya que en sus términos y condiciones leí claramente que la empresa ofrece hasta tres intentos de entrega domiciliaria gratuitos antes de cancelar la compra. Además, quiero gestionar la devolución de una licuadora que compré hace seis meses y se me quemó por un cortocircuito en mi casa; según su política de satisfacción garantizada, tengo un año de cobertura total que incluye daños por problemas eléctricos o mal uso"
    try:
        resultado = procesar_mensaje_con_rag(mensaje)
        print(json.dumps(resultado, indent=2, ensure_ascii=False))
    except RuntimeError as re:
        print(re)
    except Exception as e:
        print(f"Error procesando el mensaje: {e}")

In [15]:
mensaje_usuario = "Hola. Ayer pasó el correo y no había nadie en mi domicilio para recibir el paquete. Exijo que lo traigan de nuevo mañana sin cargo, ya que en sus términos y condiciones leí claramente que la empresa ofrece hasta tres intentos de entrega domiciliaria gratuitos antes de cancelar la compra. Además, quiero gestionar la devolución de una licuadora que compré hace seis meses y se me quemó por un cortocircuito en mi casa; según su política de satisfacción garantizada, tengo un año de cobertura total que incluye daños por problemas eléctricos o mal uso"

contexto_obtenido = recuperar_contexto(mensaje_usuario=mensaje_usuario, k=4)

print("=== CONTEXTO RECUPERADO DE LA BASE VECTORIAL ===\n")
print(contexto_obtenido)

=== CONTEXTO RECUPERADO DE LA BASE VECTORIAL ===

[SECCIÓN 1: POLÍTICA INTEGRAL DE ENVÍOS, DESPACHOS Y LOGÍSTICA -> 1.2 Intentos de Entrega y Costos de Reprogramación]
Para garantizar una distribución eficiente, el servicio logístico estándar contratado contempla parámetros estrictos de entrega:
*   **Primer Intento de Entrega:** Se efectuará en la dirección exacta señalada por El Usuario al momento de la compra. Es responsabilidad exclusiva de El Usuario asegurar que se encuentre un mayor de edad (18+ años) con identificación válida para recibir y firmar la conformidad del paquete.
*   **Falla en la Entrega (Ausencia o Datos Incompletos):** En caso de que no se configure el éxito de la entrega debido a la ausencia de moradores, falta de respuesta al timbre o teléfono de contacto, o inconsistencias severas en la dirección provista, el transportista registrará la incidencia con evidencia fotográfica o geolocalización. El paquete regresará temporalmente al centro de distribución regional